# Baselines: naive fine-tune and full retrain

Trains both baselines over P1 to P4 on a Kaggle GPU and logs them to MLflow on DagsHub.
See `docs/design.md` sections 7 and 8, `src/training/baselines.py` and `docs/baselines.md`.

**Before you run**

1. Settings: Accelerator **GPU T4 x2**, Internet **on**.
2. Add-ons, Secrets: attach `DAGSHUB_TOKEN` (the same token as `.dvc/config.local`).
   Secrets are not kept when a notebook is pushed with the Kaggle CLI, so attach it again
   after every CLI push.
3. Choose **Save & Run All (Commit)**. It runs in the background and the GPU session ends
   by itself when the notebook finishes or fails. Do not use an interactive session: it
   keeps the GPU until you stop it.

The token is only put into environment variables and is never printed. The run stops at the
first error so no GPU time is wasted. Expected time: about 22 minutes per seed.

## 1. Credentials

In [ ]:
import os

from kaggle_secrets import UserSecretsClient

token = UserSecretsClient().get_secret("DAGSHUB_TOKEN")

# DVC reads its S3 keys from these variables, and MLflow reads the other three.
os.environ["AWS_ACCESS_KEY_ID"] = token
os.environ["AWS_SECRET_ACCESS_KEY"] = token
os.environ["MLFLOW_TRACKING_URI"] = "https://dagshub.com/Akmal76/slangguard-data.mlflow"
os.environ["MLFLOW_TRACKING_USERNAME"] = "Akmal76"
os.environ["MLFLOW_TRACKING_PASSWORD"] = token

## 2. Code and data

Kaggle already has torch and transformers. `mlflow-skinny` is the small MLflow client,
enough to log to a server. `dvc[s3]` is needed to pull the data.

In [ ]:
# Pinned to the Week 2 code. Later, src/training/baselines.py was renamed to run.py,
# so do not unpin this notebook.
COMMIT = "a0a39ee"

!git clone -q https://github.com/Akmal76/slangguard.git
%cd slangguard
if COMMIT:
    !git checkout -q {COMMIT}
!git log -1 --format="code at commit %h: %s"

In [ ]:
!pip install -q mlflow-skinny==3.18.0 'dvc[s3]'

In [ ]:
!dvc pull data/periods

In [ ]:
import torch

if not torch.cuda.is_available():
    raise RuntimeError("No GPU found. Turn on a GPU accelerator in the notebook settings.")
print(torch.cuda.get_device_name(0))

## 3. Train

Seed 42 was run before (see `docs/baselines.md`). This run adds seeds 43 and 44. Each run
shows up in MLflow as `<method>-seed<seed>`. `set -e` stops everything at the first error.

In [ ]:
import os

SEEDS = [43, 44]
os.environ["SEEDS"] = " ".join(map(str, SEEDS))

In [ ]:
%%bash
set -e
for seed in $SEEDS; do
    python -m src.training.baselines --method naive --seed $seed
    python -m src.training.baselines --method full --seed $seed
done

## 4. Results

The result files are copied to `/kaggle/working/results`, so they can be downloaded with
`kaggle kernels output`.

In [ ]:
import json
import shutil
from pathlib import Path

Path("/kaggle/working/results").mkdir(exist_ok=True)
for method in ("naive", "full"):
    for seed in SEEDS:
        path = Path(f"outputs/{method}_seed{seed}.json")
        r = json.loads(path.read_text())
        print(
            f"{method} seed {seed}: AvgF1 {r['avg_f1']:.3f}, forgetting {r['forgetting']:.3f}, "
            f"P1 after step 4 {r['matrix'][-1][0]:.3f}, "
            f"{r['total_train_seconds'] / 60:.1f} min"
        )
        shutil.copy(path, "/kaggle/working/results")